## Getting exceedance times for each event

In [3]:
import pandas as pd
import numpy as np
# import event-tauc summary tables
spring_events = pd.read_csv("spring_events/spring_event_tauc_summary.csv", parse_dates=["event_start", "event_peak", "event_end", "first_exceedance_time"])
summer_events = pd.read_csv("summer_events/summer_event_tauc_summary.csv", parse_dates=["start", "peak_time", "end"])
tau = pd.read_csv("../data/shear_stress/average_total_shear_stress_corrected.csv", parse_dates=["datetime"])

# rename columns so spring and summer match
spring_events = spring_events.rename(columns={
    "event_start": "start",
    "event_peak": "peak_time",
    "event_end": "end",
})

# ensure all relevant columns have the correct type
for events in [spring_events, summer_events]:
    for col in ["start", "peak_time", "end", "first_exceedance_time"]:
        events[col] = pd.to_datetime(events[col], errors="coerce")

    events["first_exceedance_tau"] = pd.to_numeric(events["first_exceedance_tau"], errors="coerce")

tau = tau.set_index("datetime").sort_index()
tau_series = (pd.to_numeric(tau["shear_stress"], errors="coerce").groupby(level=0).mean().sort_index().dropna())

Defining functions

In [4]:
def _get_tau_window_with_boundaries(tau_series, start, end):
    """
    extract tau between start and end and interpolate tau exactly at the
    start and end boundaries if/when those times are not in the original record.
    """
    if pd.isna(start) or pd.isna(end) or start >= end:
        return pd.Series(dtype=float)

    # need one measurement before and after the event boundaries
    before = tau_series.loc[:start].tail(1)
    within = tau_series.loc[start:end]
    after = tau_series.loc[end:].head(1)

    if before.empty or after.empty:
        return pd.Series(dtype=float)

    window = pd.concat([before, within, after])
    window = window[~window.index.duplicated(keep="last")].sort_index()

    # Add exact start and end times
    new_index = window.index.union(pd.DatetimeIndex([start, end])).sort_values()
    window = window.reindex(new_index)
    # interpolate only between existing measurements
    window = window.interpolate(method="time", limit_area="inside")
    return window.loc[start:end].dropna()

def integrate_positive_excess(tau_series, tauc, start, end, max_gap="20min"):
    """
    Calculate the integral of max(tau - tauc, 0) through time and duration for which tau is above tauc;
    when the threshold is crossed between measurements, we use linear interpolation
    """
    window = _get_tau_window_with_boundaries(tau_series, start, end)

    if len(window) < 2 or pd.isna(tauc):
        return {
            "integral": np.nan,
            "duration_h": np.nan,
            "covered_h": 0.0,
            "n_skipped_intervals": 0
        }

    max_gap = pd.Timedelta(max_gap) if max_gap is not None else None
    integral = 0.0
    duration_h = 0.0
    covered_h = 0.0
    n_skipped = 0
    times = window.index
    excess = window.to_numpy(dtype=float) - float(tauc)

    for i in range(len(window) - 1):
        interval = times[i + 1] - times[i]
        dt_h = interval.total_seconds() / 3600

        if dt_h <= 0:
            continue

        # do not integrate across large data gaps
        if max_gap is not None and interval > max_gap:
            n_skipped += 1
            continue
        y0 = excess[i]
        y1 = excess[i + 1]

        if not np.isfinite(y0) or not np.isfinite(y1):
            continue
        covered_h += dt_h

        # both endpoints at or below tauc
        if y0 <= 0 and y1 <= 0:
            continue

        # both endpoints at or above tauc
        if y0 >= 0 and y1 >= 0:
            duration_h += dt_h
            integral += 0.5 * (y0 + y1) * dt_h
            continue

        # threshold crossed upward during the interval
        if y0 < 0 < y1:
            fraction_before_crossing = -y0 / (y1 - y0)
            positive_dt_h = dt_h * (1 - fraction_before_crossing)

            duration_h += positive_dt_h

            # triangle above the threshold
            integral += 0.5 * y1 * positive_dt_h
            continue

        # threshold crossed downward during the interval
        if y0 > 0 > y1:
            fraction_above = y0 / (y0 - y1)
            positive_dt_h = dt_h * fraction_above

            duration_h += positive_dt_h

            # triangle above the threshold
            integral += 0.5 * y0 * positive_dt_h

    return {
        "integral": integral,
        "duration_h": duration_h,
        "covered_h": covered_h,
        "n_skipped_intervals": n_skipped
    }

def calculate_event_exceedance_metrics(events, tau_series, max_gap="20min"):
    """
    calculate hydraulic exceedance metrics for every event.
    first_exceedance_tau is treated as the event-specific tauc50.
    """

    tau_series = tau_series.sort_index().dropna()
    output_rows = []
    for _, row in events.iterrows():
        result = {}
        start = row["start"]
        supplied_peak_time = row["peak_time"]
        end = row["end"]

        # event-specific observed critical shear stress
        tauc = row["first_exceedance_tau"]
        first_motion_time = row.get("first_exceedance_time", pd.NaT)
        result["tauc50"] = tauc

        # quality-control check
        result["first_motion_in_event"] = (pd.notna(first_motion_time) and pd.notna(start) and pd.notna(end) and start <= first_motion_time <= end)
        if (pd.isna(start) or pd.isna(end) or start >= end):
            result["calculation_status"] = "invalid event window"
            output_rows.append(result)
            continue
        if pd.isna(tauc):
            # no observed D50 transport means there is no measured tauc50
            result["calculation_status"] = "no measured tauc50"
            output_rows.append(result)
            continue
        event_tau = tau_series.loc[start:end].dropna()
        if event_tau.empty:
            result["calculation_status"] = "no tau data in event"
            output_rows.append(result)
            continue

        # peak from the complete shear-stress record
        tau_peak = event_tau.max()
        tau_peak_time_record = event_tau.idxmax()
        result["tau_peak"] = tau_peak
        result["tau_peak_time_record"] = tau_peak_time_record
        result["peak_mobility_ratio"] = (tau_peak / tauc if tauc != 0 else np.nan)
        result["max_excess_tau"] = max(tau_peak - tauc, 0)
        result["tau_reached_tauc"] = bool(tau_peak >= tauc)

        # use supplied event peak for limb separation when valid.
        # otherwise, use the maximum in the tau record.
        if (pd.notna(supplied_peak_time) and start <= supplied_peak_time <= end):
            split_peak_time = supplied_peak_time
            result["peak_time_source"] = "event table"
        else:
            split_peak_time = tau_peak_time_record
            result["peak_time_source"] = "tau record maximum"
        result["split_peak_time_used"] = split_peak_time

        # timing of observed D50 transport relative to the hydrograph peak
        if pd.notna(first_motion_time):
            result["first_motion_relative_to_peak_h"] = (first_motion_time - split_peak_time).total_seconds() / 3600
        else:
            result["first_motion_relative_to_peak_h"] = np.nan

        rising = integrate_positive_excess(tau_series=tau_series, tauc=tauc, start=start, end=split_peak_time, max_gap=max_gap)
        falling = integrate_positive_excess(tau_series=tau_series, tauc=tauc, start=split_peak_time, end=end, max_gap=max_gap)

        E_rise = rising["integral"]
        E_fall = falling["integral"]

        T_rise = rising["duration_h"]
        T_fall = falling["duration_h"]

        E_total = E_rise + E_fall
        T_total = T_rise + T_fall

        result["excess_integral_rise"] = E_rise
        result["excess_integral_fall"] = E_fall
        result["excess_integral_total"] = E_total

        result["tau_above_tauc_h_rise"] = T_rise
        result["tau_above_tauc_h_fall"] = T_fall
        result["tau_above_tauc_h_total"] = T_total

        result["mean_excess_tau"] = (E_total / T_total if T_total > 0 else np.nan) # average amount by which tau exceeded tauc while above threshold
        result["excess_fraction_rise"] = (E_rise / E_total if E_total > 0 else np.nan) # fraction of cumulative exceedance on each limb
        result["excess_fraction_fall"] = (E_fall / E_total if E_total > 0 else np.nan)

        # -1 = completely rising-limb dominated
        # +1 = completely falling-limb dominated
        result["excess_balance"] = ((E_fall - E_rise) / E_total if E_total > 0 else np.nan)

        # equivalent fractions based only on exceedance duration
        result["duration_fraction_rise"] = (T_rise / T_total if T_total > 0 else np.nan)
        result["duration_fraction_fall"] = (T_fall / T_total if T_total > 0 else np.nan)
        result["duration_balance"] = ((T_fall - T_rise) / T_total if T_total > 0 else np.nan)

        # tau-record coverage and gap checks
        covered_h = (rising["covered_h"] + falling["covered_h"])
        event_length_h = (end - start).total_seconds() / 3600
        result["tau_record_coverage_fraction"] = (covered_h / event_length_h if event_length_h > 0 else np.nan)
        result["n_skipped_tau_intervals"] = (rising["n_skipped_intervals"] + falling["n_skipped_intervals"])

        # first point in the full tau record that reaches the threshold
        tau_at_or_above = event_tau[event_tau >= tauc]
        if not tau_at_or_above.empty:
            first_record_threshold_time = tau_at_or_above.index[0]
            result["first_tau_ge_tauc_time_record"] = (first_record_threshold_time)
            if pd.notna(first_motion_time):
                result["record_threshold_minus_observed_motion_min"] = (first_record_threshold_time - first_motion_time).total_seconds() / 60
            else:
                result["record_threshold_minus_observed_motion_min"] = np.nan
        else:
            result["first_tau_ge_tauc_time_record"] = pd.NaT
            result["record_threshold_minus_observed_motion_min"] = np.nan
        result["calculation_status"] = "calculated"
        output_rows.append(result)

    metrics = pd.DataFrame(output_rows)
    return pd.concat([events.reset_index(drop=True), metrics.reset_index(drop=True)], axis=1)

Calculate exceedance metrics for each season:

In [5]:
spring_exceedance = calculate_event_exceedance_metrics(spring_events, tau_series, max_gap="20min")
summer_exceedance = calculate_event_exceedance_metrics(summer_events, tau_series, max_gap="20min")

Export as a csv

In [7]:
spring_exceedance.to_csv("spring_events/spring_event_exceedance_metrics.csv", index=False)
summer_exceedance.to_csv("summer_events/summer_event_exceedance_metrics.csv", index=False)